# Dashboard final: mockup, validación y export para Data Studio

Este notebook se encarga de preparar la entrega BI final del proyecto:
- valida que los CSVs estén en formato listo para dashboard,
- genera un mockup visual base en HTML,
- organiza los exports para Data Studio,
- y deja la estructura de archivos limpia para uso analítico.

Objetivo:
- revisar los CSVs de dashboard actuales,
- preparar un mockup visual reproducible,
- asegurar que solo queden los exports finales para Data Studio,
- documentar la validación del proyecto.

In [1]:
from pathlib import Path
import json
import pandas as pd
from html import escape

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'data' / 'dashboard').exists():
        project_root = candidate
        break

processed_dir = project_root / 'data' / 'processed'
dashboard_dir = project_root / 'data' / 'dashboard'
images_dir = project_root / 'images'
images_dir.mkdir(exist_ok=True, parents=True)

def ensure_required_file(path: Path, name: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Falta {name} en {path}')

required_dashboard = ['dashboard_ejecutivo.csv', 'dashboard_contenidos.csv', 'dashboard_predicciones.csv']
for name in required_dashboard:
    ensure_required_file(dashboard_dir / name, name)

executive = pd.read_csv(dashboard_dir / 'dashboard_ejecutivo.csv')
contenidos = pd.read_csv(dashboard_dir / 'dashboard_contenidos.csv')
predicciones = pd.read_csv(dashboard_dir / 'dashboard_predicciones.csv')
print('dashboard_ejecutivo:', executive.shape)
print('dashboard_contenidos:', contenidos.shape)
print('dashboard_predicciones:', predicciones.shape)
print('Columnas ejecutivas:', list(executive.columns))
print('Columnas contenidos:', list(contenidos.columns[:10]))

dashboard_ejecutivo: (6, 7)
dashboard_contenidos: (32000, 14)
dashboard_predicciones: (32000, 10)
Columnas ejecutivas: ['tipo', 'categoria_predicha', 'titulos', 'score_promedio', 'popularidad_media', 'rating_medio', 'contenidos_altos']
Columnas contenidos: ['id_contenido', 'titulo', 'tipo', 'generos', 'pais', 'idioma', 'anio_publicacion', 'popularidad', 'votos', 'valoracion_promedio']


## Validación de artefactos para dashboard

En esta sección se confirma que los exports finales cumplen con la estructura esperada y no quedan archivos obsoletos o duplicados para Data Studio.

In [2]:
required_columns = {
    'dashboard_ejecutivo.csv': ['tipo', 'categoria_predicha', 'titulos', 'score_promedio'],
    'dashboard_contenidos.csv': ['titulo', 'tipo', 'generos', 'pred_indice_exito'],
    'dashboard_predicciones.csv': ['titulo', 'tipo', 'score_predicho', 'categoria_predicha'],
}

for file_name, cols in required_columns.items():
    df = pd.read_csv(dashboard_dir / file_name)
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise ValueError(f'{file_name} falta columnas: {missing}')
    print(f'{file_name}: OK ({len(df)} filas, columnas requeridas presentes)')

csv_files = sorted(p.name for p in dashboard_dir.iterdir() if p.is_file() and p.suffix.lower() == '.csv')
print('CSV actuales para Data Studio:', csv_files)
assert set(csv_files).issuperset(set(required_dashboard)), 'No hay CSVs finales para Data Studio.'

dashboard_ejecutivo.csv: OK (6 filas, columnas requeridas presentes)
dashboard_contenidos.csv: OK (32000 filas, columnas requeridas presentes)
dashboard_predicciones.csv: OK (32000 filas, columnas requeridas presentes)
CSV actuales para Data Studio: ['dashboard_contenidos.csv', 'dashboard_ejecutivo.csv', 'dashboard_predicciones.csv']


In [3]:
mockup_path = images_dir / 'dashboard_streamview_mockup.html'

top_titles = predicciones.sort_values('score_predicho', ascending=False).head(5).copy()
summary = {
    'total_contenidos': int(len(contenidos)),
    'top_score': round(float(top_titles['score_predicho'].max()), 4),
    'tipo_principal': contenidos['tipo'].mode().iloc[0] if not contenidos.empty else 'N/A',
    'score_medio': round(float(predicciones['score_predicho'].mean()), 4),
}

html = f"""
<!DOCTYPE html>
<html lang=\"es\">
<head>
  <meta charset=\"utf-8\" />
  <title>StreamView Analytics - Mockup Dashboard</title>
  <style>
    body {{ font-family: Arial, sans-serif; background: #f4f7fb; margin: 0; padding: 24px; color: #1f2d3d; }}
    h1 {{ margin-bottom: 8px; }}
    .grid {{ display: grid; grid-template-columns: repeat(4, minmax(180px, 1fr)); gap: 16px; margin: 18px 0 22px; }}
    .card {{ background: white; border-radius: 12px; padding: 18px; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }}
    .kpi {{ font-size: 28px; font-weight: bold; color: #1f6feb; margin-top: 10px; }}
    .section {{ background: white; border-radius: 12px; padding: 18px; margin-top: 20px; }}
    table {{ width: 100%; border-collapse: collapse; margin-top: 12px; }}
    th, td {{ border-bottom: 1px solid #e5e7eb; padding: 10px; text-align: left; }}
    th {{ background: #eef4ff; }}
  </style>
</head>
<body>
  <h1>StreamView Analytics - Mockup Dashboard</h1>
  <p>Panel ejecutivo y predictivo para priorización de contenidos</p>
  <div class=\"grid\">
    <div class=\"card\"><div>Total de contenidos</div><div class=\"kpi\">{summary['total_contenidos']}</div></div>
    <div class=\"card\"><div>Top score</div><div class=\"kpi\">{summary['top_score']}</div></div>
    <div class=\"card\"><div>Tipo principal</div><div class=\"kpi\">{escape(str(summary['tipo_principal']))}</div></div>
    <div class=\"card\"><div>Score medio</div><div class=\"kpi\">{summary['score_medio']}</div></div>
  </div>

  <div class=\"section\">
    <h2>Top 5 prioridades de contenido</h2>
    <table>
      <tr><th>Título</th><th>Tipo</th><th>Score</th><th>Categoría</th></tr>
      {''.join(f'<tr><td>{escape(str(row["titulo"]))}</td><td>{escape(str(row["tipo"]))}</td><td>{row["score_predicho"]:.4f}</td><td>{escape(str(row["categoria_predicha"]))}</td></tr>' for _, row in top_titles.iterrows())}
    </table>
  </div>
</body>
</html>
"""

mockup_path.write_text(html, encoding='utf-8')
print(f'Mockup HTML generado en: {mockup_path}')

Mockup HTML generado en: C:\Users\admin\Documents\GitHub\visualizacion-de-datos-StreamView-Analytics\images\dashboard_streamview_mockup.html


In [4]:
# Export final para Data Studio / Looker Studio
export_files = {
    'dashboard_ejecutivo.csv': executive,
    'dashboard_contenidos.csv': contenidos,
    'dashboard_predicciones.csv': predicciones,
}
for name, df in export_files.items():
    final_path = dashboard_dir / name
    df.to_csv(final_path, index=False)
    print(f'Exportado: {final_path}')

print('La capa de dashboard quedó lista para Data Studio con archivos finales y reproducibles.')

Exportado: C:\Users\admin\Documents\GitHub\visualizacion-de-datos-StreamView-Analytics\data\dashboard\dashboard_ejecutivo.csv
Exportado: C:\Users\admin\Documents\GitHub\visualizacion-de-datos-StreamView-Analytics\data\dashboard\dashboard_contenidos.csv
Exportado: C:\Users\admin\Documents\GitHub\visualizacion-de-datos-StreamView-Analytics\data\dashboard\dashboard_predicciones.csv
La capa de dashboard quedó lista para Data Studio con archivos finales y reproducibles.


## Conclusión

La entrega final de dashboard queda lista para Data Studio con tres CSVs definitivos y un mockup visual que refleja la lógica del proyecto: panel ejecutivo, detalle de contenidos y prioridad predictiva.

La validación confirma que no faltan archivos clave y que los CSV exportados cumplen con la estructura mínima esperada para consumo BI.